<font size = "4">

**Dynamic Arrays**

- Python lists are actually implemented in the C programming language as **dynamic arrays** (often, just "arrays").

- Arrays can only store one kind of data (e.g., integers, floating-point numbers etc.). You cannot mix two data types in a single array.

- An array is a contiguous block of bytes in memory. This block is divided up into $n$-byte chunks where $n$ is the size of the relevant data type.

- Here is an array that holds six (double-precision) floating point numbers. If this were in C, each double requires 8 bytes, so each chunk corresponds to 8 individual bytes.

<div style="text-align: center;">
  <img src="files/array.png" alt="Centered image" width = "450">
  <br>
  <br>
  <figcaption>
  <font size = "1"> Miller, Randum, Yasinovskyy (Problem Solving with Algorithms and Data Structures using Python)</figcaption>
</div>

<br>



<font size = "4">

- The base address is the location in memory of where the array starts.

- For a given index, we can find the memory address of the corresponding element through:

```python
    item_address = base_address + index * size_of_object
``` 

- The calculation for the location of an object is $\mathcal{O}(1)$.

- There are some risks involved: the size of the array is fixed, so appending indefinitely can have serious consequences, like overwriting unrelated data. To prevent this, languages like C raise "Segmentation Faults"

<font size = "4">

**Python list implementation**

- Python uses an array that holds references (*pointers*) to objects.

- Python uses *over-allocation*. An array can be allocated with more space than initially needed.

- If the array is filled up, a new, bigger array is (over-)allocated and the contents of the old array are copied to the new array.

- Summary of operation counts of Python lists:

    - Accessing an item at a specific location is $\mathcal{O}(1)$.

    - Appending to the list is $\mathcal{O}(1)$ on average, but $\mathcal{O}(n)$ in the worst case (when a new array needs to be allocated).

    - Popping from the end of the list is $\mathcal{O}(1)$.

    - Deleting an item from the list is $\mathcal{O}(n)$ on average.
    
    - Inserting an item into an arbitrary position is $\mathcal{O}(n)$ on average.

<font size = "4">

**Implementing an array**

- We implement an `ArrayList` class that uses a Python list to *emulate* the array.

- In the constructor, `__init__` we initialize:

    1. `size_exponent`: this will allow us to "grow" the array as it is filled up.

    2. `max_size`: this keeps track of the allocated size of the current array.

    3. `last_index`: where the last element of the array is actually stored within the array (actually the last element is stored in `last_index - 1`)
    
    4. `my_array`: this is the actual array containing the data. We emulate it using a Python list. 

In [1]:
class ArrayList:
    def __init__(self):
        self.size_exponent = 0
        self.max_size = 0
        self.last_index = 0
        self.my_array = []

    def append(self, val):
        if self.last_index > self.max_size - 1:
            self.__resize()
        self.my_array[self.last_index] = val
        self.last_index += 1

    def __resize(self):
        new_size = 2 ** self.size_exponent
        print("new_size = ", new_size)
        new_array = [0] * new_size
        for i in range(self.max_size):
            new_array[i] = self.my_array[i]

        self.max_size = new_size
        self.my_array = new_array
        self.size_exponent += 1

    def __getitem__(self, idx):
        if idx < self.last_index:
            return self.my_array[idx]
        raise LookupError("index out of bounds")

    def __setitem__(self, idx, val):
        if idx < self.last_index:
            self.my_array[idx] = val
        raise LookupError("index out of bounds")

    def insert(self, idx, val):
        if self.last_index > self.max_size - 1:
            self.__resize()
        for i in range(self.last_index, idx - 1, -1):
            self.my_array[i + 1] = self.my_array[i]
        self.last_index += 1
        self.my_array[idx] = val

<font size = "4">

**`.append` method**

- First, checks if adding a new element would exceed maximum array size. 

- If this is the case, then call the `.__resize` method to increase the size.

- Add the value to coincide with the current value of `last_index`, then increase `last_index` by one.

**`.__resize` method**

- For this implementation, we increase the maximum array size to the next power of $2$. 

- A new array of zeros is created, and the values are copied over.


**Note:** Instead of a multiplier of $2$, other multipliers could be used. Python uses a multiplier of 1.125 plus a constant. So Python lists that increase in size lead to array sizes of $0, 4, 8, 16, 24, 32, 40, 52, 64, 76, ...$


<font size = "4">

**Cost of append**

- To analyze the cost of append, we use a technique called **amortized analysis**. 

- In this approach we average over a *sequence* of operations. 

- Let's consider building up an array by a sequence of $n$ append operations.

- If the array doubles at each step it becomes full (slightly different than what we have implemented), then it doubles when its size becomes 1, 2, 4, 8, ... Each element needs to be copied over to the new array, so the allocation cost for $n = 2^k$ is:

$$\sum_{j=0}^{ \log_2 n} 2^j$$

- Before the array-doubling, there were $n = 2^k$ individual appends at $\mathcal{O}(1)$ cost. So the total cost of $n = 2^k$ appends is:

$$T(n) = n + \sum_{j=0}^{ \log_2 n} 2^j$$

- In general, for any $n \geq 1$ we have (using the geometric series formula):

$$T(n) = n + \sum_{j=0}^{\lfloor \log_2 n \rfloor} 2^j < n + 2n = 3n$$

- The cost for $n$ appends is $\mathcal{O}(n)$. But the *cost per element* is:

$$\frac{T(n)}{n} < \frac{3n}{n} = 3$$

- So the amortized cost is $\mathcal{O}(1)$

<font size = "4">

**Accessing items**

- For `__getitem__` and `__setitem__`, notice that we don't do the calculation `item_address = base_address + index * size_of_object` 

- One reason is that it is difficult to implement.

- Another reason is that even in languages like C, this calculation is "hidden" and array elements can be accessed using similar syntax to Python. 

<font size = "4">

**Insertion**

- To insert an item at an arbitrary index, we need to (a) make sure there is enough room, and resize the array if necessary, and (b) shift all elements in the array at and after the insertion point by one index position.

- In order to avoid overwriting data, we need to start shifting from the end of the array:


<div style="text-align: center;">
  <img src="files/insertArray.png" alt="Centered image" width = "450">
  <br>
  <br>
  <figcaption>
  <font size = "1"> Miller, Randum, Yasinovskyy (Problem Solving with Algorithms and Data Structures using Python)</figcaption>
</div>

<br>

<font size = "4">

- The performance of insert is still $\mathcal{O}(n)$ in the worst case, which is inserting at position 0.

- On average, we will only need to shift $\frac{n}{2}$ elements, but this is still $\mathcal{O}(n)$, meaning the average cost has the same order.


<font size = "4">

We can see our array class in action using a sequence of appends:

In [2]:
x = ArrayList()

for i in range(1,11):
    print(f"i = {i}")
    x.append(i)
    print(x.my_array)

i = 1
new_size =  1
[1]
i = 2
new_size =  2
[1, 2]
i = 3
new_size =  4
[1, 2, 3, 0]
i = 4
[1, 2, 3, 4]
i = 5
new_size =  8
[1, 2, 3, 4, 5, 0, 0, 0]
i = 6
[1, 2, 3, 4, 5, 6, 0, 0]
i = 7
[1, 2, 3, 4, 5, 6, 7, 0]
i = 8
[1, 2, 3, 4, 5, 6, 7, 8]
i = 9
new_size =  16
[1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 0, 0, 0, 0, 0, 0]
i = 10
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 0, 0, 0, 0, 0, 0]


<font size = "4">

**Note:** When resizing an actual array, the additional elements are not zero, but whatever integer corresponds to that particular block of memory. There is no point in initializing the values to zero if they aren't meaningful.

We can see this using `np.empty`, which creates an "empty" array that actually contains whatever random information was stored there before initialization:

In [3]:
import numpy as np

print(np.empty(5,))


[ 1.    2.75  6.   10.75 17.  ]


<font size = "4">

But sometimes it does correspond to zeros, depending on what memory is available on the computer and your operating system behavior:

In [4]:
help(np.empty)

Help on built-in function empty in module numpy:

empty(...)
    empty(shape, dtype=float, order='C', *, like=None)

    Return a new array of given shape and type, without initializing entries.

    Parameters
    ----------
    shape : int or tuple of int
        Shape of the empty array, e.g., ``(2, 3)`` or ``2``.
    dtype : data-type, optional
        Desired output data-type for the array, e.g, `numpy.int8`. Default is
        `numpy.float64`.
    order : {'C', 'F'}, optional, default: 'C'
        Whether to store multi-dimensional data in row-major
        (C-style) or column-major (Fortran-style) order in
        memory.
    like : array_like, optional
        Reference object to allow the creation of arrays which are not
        NumPy arrays. If an array-like passed in as ``like`` supports
        the ``__array_function__`` protocol, the result will be defined
        by it. In this case, it ensures the creation of an array object
        compatible with that passed in via thi

In [8]:
print(np.empty(11,))

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


<font size = "4">

We can try to get some hint as to how the array size grows when we create lists with larger number of elements.

The `sys.getsizeof` function returns the size of an object in bytes.

In [7]:
import sys
print(sys.getsizeof(0))

print(sys.getsizeof(0.))

28
24


In [8]:
import sys

print(sys.getsizeof(5.4))
print(sys.getsizeof(-1.0))
print(sys.getsizeof(0.0))
print(sys.getsizeof(0))

24
24
24
28


In [9]:
print(sys.getsizeof([]))
print(sys.getsizeof([3.1415]))
print(sys.getsizeof([3.1415, 2.189]))
print(sys.getsizeof([3.1415, 2.189, 0.0]))
print(sys.getsizeof([3.1415, 2.189, 0.1, 0.1]))
print(sys.getsizeof([3.1415, 2.189, 0.1, 0.1, 9.87]))
print(sys.getsizeof([3.1415, 2.189, 0.1, 0.1, 9.87, 10.2]))
print(sys.getsizeof([3.1415, 2.189, 0.1, 0.1, 9.87, 10.2, 15.3]))
print(sys.getsizeof([3.1415, 2.189, 0.1, 0.1, 9.87, 10.2, 15.3, 1.0]))
print(sys.getsizeof([3.1415, 2.189, 0.1, 0.1, 9.87, 10.2, 15.3, 1.0, 2.0]))

56
64
72
88
88
104
104
120
120
136
